In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week5-assignment-2"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
# Use the dataset given in HDFS(path: /public/trendytech/retail_db/customers)
# cust_id,cust_fname,cust_lname,cust_email,cust_password,cust_street,cust_city,cust_state,cust_zipcode
# 1,Richard,Hernandez,XXXXXXXXX,XXXXXXXXX,6303 Heather Plaza,Brownsville,TX,78521

In [3]:
! hadoop fs -ls /public/trendytech/retail_db/customers

Found 1 items
-rw-r--r--   3 itv005857 supergroup     953719 2023-04-26 16:47 /public/trendytech/retail_db/customers/part-00000


In [4]:
!hadoop fs -head /public/trendytech/retail_db/customers/part-00000

1,Richard,Hernandez,XXXXXXXXX,XXXXXXXXX,6303 Heather Plaza,Brownsville,TX,78521
2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126
3,Ann,Smith,XXXXXXXXX,XXXXXXXXX,3422 Blue Pioneer Bend,Caguas,PR,00725
4,Mary,Jones,XXXXXXXXX,XXXXXXXXX,8324 Little Common,San Marcos,CA,92069
5,Robert,Hudson,XXXXXXXXX,XXXXXXXXX,"10 Crystal River Mall ",Caguas,PR,00725
6,Mary,Smith,XXXXXXXXX,XXXXXXXXX,3151 Sleepy Quail Promenade,Passaic,NJ,07055
7,Melissa,Wilcox,XXXXXXXXX,XXXXXXXXX,9453 High Concession,Caguas,PR,00725
8,Megan,Smith,XXXXXXXXX,XXXXXXXXX,3047 Foggy Forest Plaza,Lawrence,MA,01841
9,Mary,Perez,XXXXXXXXX,XXXXXXXXX,3616 Quaking Street,Caguas,PR,00725
10,Melissa,Smith,XXXXXXXXX,XXXXXXXXX,8598 Harvest Beacon Plaza,Stafford,VA,22554
11,Mary,Huffman,XXXXXXXXX,XXXXXXXXX,3169 Stony Woods,Caguas,PR,00725
12,Christopher,Smith,XXXXXXXXX,XXXXXXXXX,5594 Jagged Embers By-pass,San Antonio,TX,78227
13,Mary,Baldwin,XXXXXXXXX,XXXXXXXXX,7922 Iron Oak Gardens,Caguas,PR,00725
14,Katherine

In [5]:
cust_df = spark.read \
.format("csv") \
.option("inferSchema","true") \
.load("/public/trendytech/retail_db/customers")

In [6]:
cust_df.show(5)

+---+-------+---------+---------+---------+--------------------+-----------+---+-----+
|_c0|    _c1|      _c2|      _c3|      _c4|                 _c5|        _c6|_c7|  _c8|
+---+-------+---------+---------+---------+--------------------+-----------+---+-----+
|  1|Richard|Hernandez|XXXXXXXXX|XXXXXXXXX|  6303 Heather Plaza|Brownsville| TX|78521|
|  2|   Mary|  Barrett|XXXXXXXXX|XXXXXXXXX|9526 Noble Embers...|  Littleton| CO|80126|
|  3|    Ann|    Smith|XXXXXXXXX|XXXXXXXXX|3422 Blue Pioneer...|     Caguas| PR|  725|
|  4|   Mary|    Jones|XXXXXXXXX|XXXXXXXXX|  8324 Little Common| San Marcos| CA|92069|
|  5| Robert|   Hudson|XXXXXXXXX|XXXXXXXXX|10 Crystal River ...|     Caguas| PR|  725|
+---+-------+---------+---------+---------+--------------------+-----------+---+-----+
only showing top 5 rows



In [7]:
cust_df.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: integer (nullable = true)



In [8]:
## cols cust_id,cust_fname,cust_lname,cust_email,cust_password,cust_street,cust_city,cust_state,cust_zipcode

In [9]:
cust_df1 = cust_df.withColumnRenamed("_c0","cust_id") \
.withColumnRenamed("_c1","cust_fname") \
.withColumnRenamed("_c2","cust_lname") \
.withColumnRenamed("_c3","cust_email") \
.withColumnRenamed("_c4","cust_password") \
.withColumnRenamed("_c5","cust_street") \
.withColumnRenamed("_c6","cust_city") \
.withColumnRenamed("_c7","cust_state") \
.withColumnRenamed("_c8","cust_zipcode") \

In [10]:
cust_df1.show(5)

+-------+----------+----------+----------+-------------+--------------------+-----------+----------+------------+
|cust_id|cust_fname|cust_lname|cust_email|cust_password|         cust_street|  cust_city|cust_state|cust_zipcode|
+-------+----------+----------+----------+-------------+--------------------+-----------+----------+------------+
|      1|   Richard| Hernandez| XXXXXXXXX|    XXXXXXXXX|  6303 Heather Plaza|Brownsville|        TX|       78521|
|      2|      Mary|   Barrett| XXXXXXXXX|    XXXXXXXXX|9526 Noble Embers...|  Littleton|        CO|       80126|
|      3|       Ann|     Smith| XXXXXXXXX|    XXXXXXXXX|3422 Blue Pioneer...|     Caguas|        PR|         725|
|      4|      Mary|     Jones| XXXXXXXXX|    XXXXXXXXX|  8324 Little Common| San Marcos|        CA|       92069|
|      5|    Robert|    Hudson| XXXXXXXXX|    XXXXXXXXX|10 Crystal River ...|     Caguas|        PR|         725|
+-------+----------+----------+----------+-------------+--------------------+-----------

In [11]:
cust_df1.printSchema()

root
 |-- cust_id: integer (nullable = true)
 |-- cust_fname: string (nullable = true)
 |-- cust_lname: string (nullable = true)
 |-- cust_email: string (nullable = true)
 |-- cust_password: string (nullable = true)
 |-- cust_street: string (nullable = true)
 |-- cust_city: string (nullable = true)
 |-- cust_state: string (nullable = true)
 |-- cust_zipcode: integer (nullable = true)



In [12]:
## 3.1. Find the total number of customers in each state.

In [13]:
state_cust_count = cust_df1.select("cust_state","cust_id").groupBy("cust_state").count().orderBy("cust_state")

In [14]:
state_cust_count.show()

+----------+-----+
|cust_state|count|
+----------+-----+
|        AL|    3|
|        AR|   12|
|        AZ|  213|
|        CA| 2012|
|        CO|  122|
|        CT|   73|
|        DC|   42|
|        DE|   23|
|        FL|  374|
|        GA|  169|
|        HI|   87|
|        IA|    5|
|        ID|    9|
|        IL|  523|
|        IN|   40|
|        KS|   29|
|        KY|   35|
|        LA|   63|
|        MA|  113|
|        MD|  164|
+----------+-----+
only showing top 20 rows



In [15]:
## Find the top 5 most common last names among the customers

In [16]:
last_name_count = cust_df1.select("cust_lname").groupBy("cust_lname").count()

In [17]:
last_name_count.orderBy("count",ascending=False).show(5)

+----------+-----+
|cust_lname|count|
+----------+-----+
|     Smith| 4626|
|   Johnson|   76|
|  Williams|   69|
|     Jones|   65|
|     Brown|   62|
+----------+-----+
only showing top 5 rows



In [18]:
##  Check whether there are any customers whose zip codes are not valid (i.e., not equal to 5 digits).

In [20]:
from pyspark.sql.functions import length
invalid_zip = cust_df1.filter(length("cust_zipcode") < 5).select("cust_id","cust_zipcode")

In [21]:
from pyspark.sql.functions import length
invalidzip = cust_df1.filter(length("cust_zipcode") < 5).select("cust_id","cust_zipcode").count()

In [22]:
invalid_zip.count()

5191

In [23]:
print(invalidzip)

5191


In [24]:
invalid_zip.show(5)

+-------+------------+
|cust_id|cust_zipcode|
+-------+------------+
|      3|         725|
|      5|         725|
|      6|        7055|
|      7|         725|
|      8|        1841|
+-------+------------+
only showing top 5 rows



In [25]:
from pyspark.sql.functions import length

valid_zip = cust_df1.filter(length("cust_zipcode") == 5).select("cust_id","cust_zipcode").count()
print(valid_zip)

7244


In [26]:
# Find the number of customers from each city in the state of California(CA).

In [27]:
california_state_cust = cust_df1.filter("cust_state = 'CA'").select("cust_city","cust_id").groupBy("cust_city").count()

In [28]:
california_state_cust.show()

+-------------+-----+
|    cust_city|count|
+-------------+-----+
|       Corona|   14|
|    Pittsburg|    4|
|      Compton|   19|
|    Palo Alto|    6|
|      Hanford|    9|
|      Anaheim|   19|
|       Folsom|    6|
|         Napa|    8|
|     Temecula|    6|
|       Reseda|    6|
|    Encinitas|   17|
|    Oceanside|   24|
|    Cupertino|    9|
|      Oakland|    3|
|        Davis|    9|
|      Fontana|   18|
|Mission Viejo|   26|
|       Madera|    5|
|    Elk Grove|   10|
|  Bakersfield|   41|
+-------------+-----+
only showing top 20 rows



In [29]:
spark.sql("use itv024128")

""


In [30]:
spark.sql("show tables")

database,tableName,isTemporary
itv024128,groceries,false
itv024128,groceries_ext,false
itv024128,groceries_ext_json,false
itv024128,groceries_json,false
itv024128,orders_ext,false


In [31]:
cust_df1.createOrReplaceTempView("customers")

In [32]:
spark.sql("select count(*) from customers")

count(1)
12435


In [33]:
spark.sql("select * from customers limit 5").show()

+-------+----------+----------+----------+-------------+--------------------+-----------+----------+------------+
|cust_id|cust_fname|cust_lname|cust_email|cust_password|         cust_street|  cust_city|cust_state|cust_zipcode|
+-------+----------+----------+----------+-------------+--------------------+-----------+----------+------------+
|      1|   Richard| Hernandez| XXXXXXXXX|    XXXXXXXXX|  6303 Heather Plaza|Brownsville|        TX|       78521|
|      2|      Mary|   Barrett| XXXXXXXXX|    XXXXXXXXX|9526 Noble Embers...|  Littleton|        CO|       80126|
|      3|       Ann|     Smith| XXXXXXXXX|    XXXXXXXXX|3422 Blue Pioneer...|     Caguas|        PR|         725|
|      4|      Mary|     Jones| XXXXXXXXX|    XXXXXXXXX|  8324 Little Common| San Marcos|        CA|       92069|
|      5|    Robert|    Hudson| XXXXXXXXX|    XXXXXXXXX|10 Crystal River ...|     Caguas|        PR|         725|
+-------+----------+----------+----------+-------------+--------------------+-----------

In [34]:
# Find the total number of customers in each state.

In [35]:
spark.sql("select cust_state,count(*) from customers group by cust_state order by cust_state")

cust_state,count(1)
AL,3
AR,12
AZ,213
CA,2012
CO,122
CT,73
DC,42
DE,23
FL,374
GA,169


In [36]:
#Find the top 5 most common last names among the customers.

In [37]:
spark.sql("select cust_lname, count(*) as count from customers group by cust_lname order by count desc limit 5")

cust_lname,count
Smith,4626
Johnson,76
Williams,69
Jones,65
Brown,62


In [38]:
#Check whether there are any customers whose zip codes are not valid

In [39]:
spark.sql("select count(*) from customers where length(cust_zipcode) != 5 ")

count(1)
5191


In [40]:
# Count the number of customers who have valid zip codes

In [41]:
spark.sql("select count(*) from customers where length(cust_zipcode) = 5 ")

count(1)
7244


In [42]:
#Find the number of customers from each city in the state of California(CA).

In [43]:
spark.sql("select cust_city, count(*) from customers where cust_state ='CA' group by cust_city")

cust_city,count(1)
Corona,14
Pittsburg,4
Compton,19
Palo Alto,6
Hanford,9
Anaheim,19
Folsom,6
Napa,8
Temecula,6
Reseda,6
